# YZM 513 — Week 6 Lab: First Steps on the GPU with CuPy

**İstanbul Medeniyet Üniversitesi — AI Engineering M.Sc. (Fall 2026)**

Goals for today:
1. Inspect the GPU that Colab gives you (`nvidia-smi`).
2. Benchmark NumPy (CPU) vs CuPy (GPU) on matmuls of increasing size and **find the crossover point**.
3. See when the **transfer overhead** dominates (the "transfer trap").
4. Connect to the course: cost ladder, Amdahl's amortisation rule.

> Run cells top to bottom (**Runtime → Run all**). Where you see 📝, copy the printed
> numbers into the results table in the markdown cell that follows.

## 0. Setup

Check the GPU and install CuPy. Colab's T4 runs CUDA 12.x, so we need the matching CuPy build.
If Colab assigns you a different GPU, everything still works — note the name for Exercise 1.

In [ ]:
!nvidia-smi

In [ ]:
# Install CuPy matching Colab's CUDA version (T4 => CUDA 12.x)
!pip install -q cupy-cuda12x

import cupy as cp
import numpy as np
print("CuPy", cp.__version__, "| CUDA", cp.cuda.runtime.runtimeGetVersion())
print("GPU:", cp.cuda.runtime.getDeviceProperties(0).get(b'name', b'?').decode())

## Exercise 1 — Meet your GPU (5 min)

Answer from the output above (and `cp.cuda.runtime.getDeviceProperties`):

| Question | Your answer |
|---|---|
| GPU name | |
| Total VRAM (GB) | |
| CUDA capability (e.g. 7.5) | |
| MultiProcessor (SM) count | |
| Why does SM count matter for SIMT? (one sentence) | |

In [ ]:
# Device properties, programmatically
props = cp.cuda.runtime.getDeviceProperties(0)
for k in [b'name', b'totalGlobalMem', b'multiProcessorCount', b'major', b'minor', b'clockRate']:
    print(k.decode(), '=', props.get(k))

# compare with what nvidia-smi reported -- any differences? (hint: clock units)

## Exercise 2 — Matmul sweep: NumPy vs CuPy (10 min)

We time three things at each size `n`:
- **NumPy** (CPU, multithreaded BLAS — remember Week 2!)
- **CuPy compute only** (data already on GPU — CUDA events give accurate GPU-side timing)
- **CuPy end-to-end** (`asarray` upload + matmul + `get()` download)

Prediction first: 📝 *at what n do you expect the GPU to win?*

In [ ]:
def time_numpy_matmul(n, iters=5):
    a = np.random.rand(n, n).astype(np.float32)
    b = np.random.rand(n, n).astype(np.float32)
    np.dot(a, b)  # warmup
    t0 = time.perf_counter()
    for _ in range(iters):
        c = np.dot(a, b)
    return (time.perf_counter() - t0) / iters

def time_cupy_matmul(n, iters=20, include_transfer=False):
    a_cpu = np.random.rand(n, n).astype(np.float32)
    b_cpu = np.random.rand(n, n).astype(np.float32)
    a = cp.asarray(a_cpu); b = cp.asarray(b_cpu)  # upload
    cp.dot(a, b)                                    # warmup
    start = cp.cuda.Event(); end = cp.cuda.Event()
    start.record()
    for _ in range(iters):
        c = cp.dot(a, b)
    end.record(); end.synchronize()
    gpu_only = cp.cuda.get_elapsed_time(start, end) / iters / 1000  # seconds
    if not include_transfer:
        return gpu_only
    t0 = time.perf_counter()
    a2 = cp.asarray(a_cpu); b2 = cp.asarray(b_cpu)
    c2 = cp.dot(a2, b2)
    result = c2.get()
    return time.perf_counter() - t0

In [ ]:
import time

sizes = [256, 512, 1024, 2048, 4096, 8192]
print(f"{'n':>6} | {'NumPy (s)':>10} | {'CuPy GPU (s)':>12} | {'CuPy end-to-end (s)':>20} | {'speedup (compute)':>18}")
print("-" * 80)
rows = []
for n in sizes:
    t_np  = time_numpy_matmul(n)
    t_gpu = time_cupy_matmul(n, include_transfer=False)
    t_e2e = time_cupy_matmul(n, include_transfer=True)
    rows.append((n, t_np, t_gpu, t_e2e))
    print(f"{n:>6} | {t_np:>10.4f} | {t_gpu:>12.6f} | {t_e2e:>20.6f} | {t_np/t_gpu:>17.1f}x")

### 📝 Results table — copy your numbers here

| n | NumPy (s) | CuPy compute (s) | CuPy end-to-end (s) | Speedup (compute-only) |
|---|---|---|---|---|
| 256 | | | | |
| 512 | | | | |
| 1024 | | | | |
| 2048 | | | | |
| 4096 | | | | |
| 8192 | | | | |

**Questions:**
1. At which `n` does the GPU first win on **compute-only** timing? On **end-to-end**?
2. Explain the gap between the two columns using the *cost ladder* (PCIe ≈ 25 GB/s vs HBM ≈ 300 GB/s).
3. For matmul, compute time grows like n³ but transfer time grows like n² — why does the GPU *inevitably* win at large n? (Amdahl-flavoured argument.)

## Exercise 3 — Element-wise ops on 10M elements (5 min)

ReLU is the simplest data-parallel op: one thread per element.
NumPy releases the GIL and vectorises; the GPU runs 10M threads. Who wins, and by how much?

In [ ]:
n = 10_000_000
x_cpu = np.random.randn(n).astype(np.float32)

# NumPy
x_cpu_copy = x_cpu.copy()
np.maximum(x_cpu_copy, 0, out=x_cpu_copy)          # warmup
t0 = time.perf_counter()
for _ in range(20):
    np.maximum(x_cpu_copy, 0, out=x_cpu_copy)
t_np = (time.perf_counter() - t0) / 20

# CuPy
x_gpu = cp.asarray(x_cpu)
cp.maximum(x_gpu, 0, out=x_gpu)                     # warmup
start = cp.cuda.Event(); end = cp.cuda.Event()
start.record()
for _ in range(20):
    cp.maximum(x_gpu, 0, out=x_gpu)
end.record(); end.synchronize()
t_gpu = cp.cuda.get_elapsed_time(start, end) / 20 / 1000

print(f"NumPy: {t_np*1e3:.3f} ms | CuPy (on-device): {t_gpu*1e3:.3f} ms | speedup: {t_np/t_gpu:.1f}x")
print("Memory bandwidth check (approx):", f"{n*4/t_gpu/1e9:.0f} GB/s used — vs T4 peak ~320 GB/s")

## Exercise 4 — The transfer trap (5 min)

Time `cp.asarray` (CPU→GPU upload) alone as a function of array size, and compare with
a cheap on-GPU op. This is the *granularity* question from Week 5, transplanted to PCIe.

In [ ]:
for mb in [0.004, 0.4, 4, 40, 400]:
    n = int(mb * 1e6 / 4)
    x = np.random.rand(n).astype(np.float32)
    t0 = time.perf_counter()
    for _ in range(20):
        x_gpu = cp.asarray(x)
    cp.cuda.Stream.null.synchronize()
    t_up = (time.perf_counter() - t0) / 20
    bw = n * 4 / t_up / 1e9
    print(f"{mb:>7.3f} MB | upload {t_up*1e6:>8.1f} us | effective {bw:>6.2f} GB/s")

# The trap: transferring a tiny array costs ~a fixed ~10-50 us round trip.
# If your compute is 5 us, you just made everything 10x SLOWER than the CPU.

**Rule to remember:** upload once → compute a lot → download once.
Never move data back and forth inside a loop (this will come back in Week 9's `pin_memory`/`non_blocking`).

## Exercise 5 — Bonus: SVD on the GPU (5 min)

Callback to Project 1 (the HP-SS engine): same linear algebra, different backend.
`cp.linalg.svd` calls cuSOLVER instead of LAPACK. How does it scale with size?

In [ ]:
for n in [1000, 2000, 4000]:
    a = np.random.rand(n, n).astype(np.float32)
    t0 = time.perf_counter(); np.linalg.svd(a); t_np = time.perf_counter() - t0
    a_gpu = cp.asarray(a)
    start = cp.cuda.Event(); end = cp.cuda.Event()
    start.record(); cp.linalg.svd(a_gpu); end.record(); end.synchronize()
    t_gpu = cp.cuda.get_elapsed_time(start, end) / 1000
    print(f"n={n:>5} | NumPy {t_np:>7.3f} s | CuPy {t_gpu:>7.3f} s | speedup {t_np/t_gpu:>5.1f}x")

## Wrap-up (to submit)

1. Your completed results tables (Ex. 2 and the n at which GPU wins).
2. One sentence: *"On a T4, GPU is worthwhile when … because …"* (use numbers from Ex. 2/4).
3. One sentence connecting this lab to the **four questions** (split / assign / move / wait) — what is the GPU's answer to each?